Ce notebook génère **3 fichiers CSV** utilisés par `ComparaisonCS_final.ipynb` :
| Fichier | Contenu | Nb features |
|---|---|---|
| `dfbase.csv` | Variables d'origine + encodage quali | ~p |
| `dfpoly.csv` | dfbase + termes X² et X³ | ~2p |
| `dfinter.csv` | dfbase + interactions d'ordre 2 | ~p(p-1)/2 |
| `dffull.csv` | dfbase + termes X² et X³ + interactions d'ordre 2 | ~p(p+5)/2 |

> ⚙️ **Chemin de sauvegarde** : modifiez `SORTIE` dans la cellule suivante si nécessaire.

In [2]:
import pandas as pd
import numpy as np
from patsy import dmatrix

# Chemin de sortie des CSV (relatif à ce notebook)
# Par défaut : FICHIERS FINAUX/ (là où ComparaisonCS_final.ipynb les attend)
SORTIE = "../Session3/"

## 1. Chargement et exploration

In [61]:
#don = pd.read_csv("https://regression-avec-python.github.io/donnees/SAh.csv",
#                  header=0, sep=",")
### import fichier csv
# don = pd.read_csv("spambase/spambase.data", header=None, sep=",")

### import fichier texte :
don = pd.read_table("ozone.txt", header=0, sep=";")

don.head()

,Date,O3,T12,T15,Ne12,N12,S12,E12,W12,Vx,O3v,nebulosite,vent
0,19960422,63.6,13.4,15.0,7,0,0,3,0,9.35,95.6,NUAGE,EST
1,19960429,89.6,15.0,15.7,4,3,0,0,0,5.40,100.2,SOLEIL,NORD
2,19960506,79.0,7.9,10.1,8,0,0,7,0,19.30,105.6,NUAGE,EST
3,19960514,81.2,13.1,11.7,7,7,0,0,0,12.60,95.2,NUAGE,NORD
4,19960521,88.0,14.1,16.0,6,0,0,0,6,-20.30,82.8,NUAGE,OUEST


In [62]:
# Si fichier de données sans noms de colonnes
# 2. Renommer la première colonne en "Y"
# column_names = [f'X{i}' for i in range(len(don.columns) - 1)] + ['Y']
# don.columns = column_names
# don.head()

In [63]:
# Si on souhaite retirer des variables inutiles du df (ex : date)
don = don.drop(columns=["Date"])
don.head(3)

,O3,T12,T15,Ne12,N12,S12,E12,W12,Vx,O3v,nebulosite,vent
0,63.6,13.4,15.0,7,0,0,3,0,9.35,95.6,NUAGE,EST
1,89.6,15.0,15.7,4,3,0,0,0,5.40,100.2,SOLEIL,NORD
2,79.0,7.9,10.1,8,0,0,7,0,19.30,105.6,NUAGE,EST


In [64]:
# Modifier nom variable à prédire
don.rename(columns={"O3": "Y"}, inplace=True)
print(f"Dimensions      : {don.shape[0]} observations x {don.shape[1]} colonnes")
print(f"Proportion Y=1  : {don.Y.mean():.1%} ({int(don.Y.sum())} malades / {len(don)} individus)")
print(f"Types de variables :")
print(don.dtypes)
don.describe(include='all')

Dimensions      : 50 observations x 12 colonnes
Proportion Y=1  : 8630.0% (4314 malades / 50 individus)
Types de variables :
Y             float64
T12           float64
T15           float64
Ne12            int64
N12             int64
S12             int64
E12             int64
W12             int64
Vx            float64
O3v           float64
nebulosite     object
vent           object
dtype: object


,Y,T12,T15,Ne12,N12,S12,E12,W12,Vx,O3v,nebulosite,vent
count,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.00000,50.000000,50,50
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,4
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NUAGE,OUEST
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28,18
mean,86.300000,20.320000,21.390000,5.020000,0.940000,0.580000,1.360000,1.540000,-0.83120,84.280000,NaN,NaN
std,23.900824,4.674638,4.980544,2.535382,2.122859,1.499524,2.229304,2.451239,13.59233,24.953868,NaN,NaN
min,41.800000,7.900000,10.100000,0.000000,0.000000,0.000000,0.000000,0.000000,-27.06000,38.000000,NaN,NaN
25%,66.600000,17.400000,18.150000,3.000000,0.000000,0.000000,0.000000,0.000000,-10.80000,63.400000,NaN,NaN
50%,83.900000,19.750000,21.050000,6.000000,0.000000,0.000000,0.000000,0.000000,-3.25500,83.400000,NaN,NaN
75%,102.200000,24.325000,25.700000,7.000000,0.000000,0.000000,3.000000,2.750000,9.35000,102.750000,NaN,NaN


## 2. Séparation X/Y et encodage des variables qualitatives

Je renomme la variable d'intérêt Y, je mets en général les X d'un côté et je regarde les types de variables afin de recoder les variables qualitatives

In [65]:
X = don.drop(columns=["Y"])
Y = don[["Y"]]

Xquanti = X.select_dtypes(exclude=['object'])
Xquali  = X.select_dtypes(include=['object'])
print("Variables quantitatives :", list(Xquanti.columns))
print("Variables qualitatives  :", list(Xquali.columns))

# Encodage one-hot (indicatrices) : drop_first=True évite la colinéarité (supprime la modalité de référence)
# Pour SAh : famhist (Present/Absent) -> famhist_Present = 1 ou 0
# Encodage one-hot (uniquement si Xquali n'est pas vide)
if not Xquali.empty:
    XqualiD = pd.get_dummies(Xquali, drop_first=True, dtype=float)
    print("Colonne(s) après encodage :", list(XqualiD.columns))
else:
    XqualiD = pd.DataFrame()  # DataFrame vide si aucune variable qualitative
    print("Aucune variable qualitative à encoder.")

# Base commune : quantitatives + qualitatives encodées
Xbase = pd.concat([Xquanti, XqualiD], axis=1)
print(f"\nNombre de variables dans Xbase : {Xbase.shape[1]}")

Xbase.head()

Variables quantitatives : ['T12', 'T15', 'Ne12', 'N12', 'S12', 'E12', 'W12', 'Vx', 'O3v']
Variables qualitatives  : ['nebulosite', 'vent']
Colonne(s) après encodage : ['nebulosite_SOLEIL', 'vent_NORD', 'vent_OUEST', 'vent_SUD']

Nombre de variables dans Xbase : 13


,T12,T15,Ne12,N12,S12,E12,W12,Vx,O3v,nebulosite_SOLEIL,vent_NORD,vent_OUEST,vent_SUD
0,13.4,15.0,7,0,0,3,0,9.35,95.6,0.0,0.0,0.0,0.0
1,15.0,15.7,4,3,0,0,0,5.40,100.2,1.0,1.0,0.0,0.0
2,7.9,10.1,8,0,0,7,0,19.30,105.6,0.0,0.0,0.0,0.0
3,13.1,11.7,7,7,0,0,0,12.60,95.2,0.0,1.0,0.0,0.0
4,14.1,16.0,6,0,0,0,6,-20.30,82.8,0.0,0.0,1.0,0.0


## 3. `dfbase.csv` — Variables d'origine encodées

In [66]:
dfbase = pd.concat([Xbase, Y], axis=1)
print(f"dfbase : {dfbase.shape[0]} obs x {dfbase.shape[1]-1} features + Y")
dfbase.head(3)

dfbase : 50 obs x 13 features + Y


,T12,T15,Ne12,N12,S12,E12,W12,Vx,O3v,nebulosite_SOLEIL,vent_NORD,vent_OUEST,vent_SUD,Y
0,13.4,15.0,7,0,0,3,0,9.35,95.6,0.0,0.0,0.0,0.0,63.6
1,15.0,15.7,4,3,0,0,0,5.40,100.2,1.0,1.0,0.0,0.0,89.6
2,7.9,10.1,8,0,0,7,0,19.30,105.6,0.0,0.0,0.0,0.0,79.0


In [67]:
dfbase.to_csv(SORTIE + "dfbase.csv", index=False)
print("Sauvegarde OK :", SORTIE + "dfbase.csv")
dfbase.head()

Sauvegarde OK : ../Session3/dfbase.csv


,T12,T15,Ne12,N12,S12,E12,W12,Vx,O3v,nebulosite_SOLEIL,vent_NORD,vent_OUEST,vent_SUD,Y
0,13.4,15.0,7,0,0,3,0,9.35,95.6,0.0,0.0,0.0,0.0,63.6
1,15.0,15.7,4,3,0,0,0,5.40,100.2,1.0,1.0,0.0,0.0,89.6
2,7.9,10.1,8,0,0,7,0,19.30,105.6,0.0,0.0,0.0,0.0,79.0
3,13.1,11.7,7,7,0,0,0,12.60,95.2,0.0,1.0,0.0,0.0,81.2
4,14.1,16.0,6,0,0,0,6,-20.30,82.8,0.0,0.0,1.0,0.0,88.0


## 4. `dfpoly.csv` — Variables + termes polynomiaux (X², X³)
On ajoute les carrés et cubes des variables **quantitatives** uniquement.

In [68]:
X2 = Xquanti ** 2; X2 = X2.add_suffix("_sq")
X3 = Xquanti ** 3; X3 = X3.add_suffix("_cu")
Xpoly = pd.concat([Xbase, X2, X3], axis=1)
dfpoly = pd.concat([Xpoly, Y], axis=1)
print(f"dfpoly : {dfpoly.shape[0]} obs x {dfpoly.shape[1]-1} features + Y")
dfpoly.head(3)

dfpoly : 50 obs x 31 features + Y


,T12,T15,Ne12,N12,S12,E12,W12,Vx,O3v,nebulosite_SOLEIL,...,T12_cu,T15_cu,Ne12_cu,N12_cu,S12_cu,E12_cu,W12_cu,Vx_cu,O3v_cu,Y
0,13.4,15.0,7,0,0,3,0,9.35,95.6,0.0,...,2406.104,3375.000,343,0,0,27,0,817.400375,873722.816,63.6
1,15.0,15.7,4,3,0,0,0,5.40,100.2,1.0,...,3375.000,3869.893,64,27,0,0,0,157.464000,1006012.008,89.6
2,7.9,10.1,8,0,0,7,0,19.30,105.6,0.0,...,493.039,1030.301,512,0,0,343,0,7189.057000,1177583.616,79.0


In [69]:
dfpoly.to_csv(SORTIE + "dfpoly.csv", index=False)
print("Sauvegarde OK :", SORTIE + "dfpoly.csv")

Sauvegarde OK : ../Session3/dfpoly.csv


## 5. `dfinter.csv` — Variables + interactions d'ordre 2
On ajoute tous les produits croisés entre variables via la formule patsy `(X1+X2+...)**2`.  
Cela modélise l'effet combiné de deux variables (ex. : âge × tabac).

In [70]:
nomsvar = list(don.columns.difference(["Y"]))
formuleI = "~1+(" + "+".join(nomsvar) + ")**2"  # effetsprincipaux et interactions d'ordre 2
Xinter = dmatrix(formuleI, don, return_type="dataframe").iloc[:, 1:]  # supprime l'intercept
dfinter = pd.concat([Xinter, Y], axis=1)
print(f"dfinter : {dfinter.shape[0]} obs x {dfinter.shape[1]-1} features + Y")
dfinter.head(3)

dfinter : 50 obs x 88 features + Y


,nebulosite[T.SOLEIL],vent[T.NORD],vent[T.OUEST],vent[T.SUD],nebulosite[T.SOLEIL]:vent[T.NORD],nebulosite[T.SOLEIL]:vent[T.OUEST],nebulosite[T.SOLEIL]:vent[T.SUD],E12,E12:nebulosite[T.SOLEIL],E12:vent[T.NORD],...,S12:T15,S12:Vx,S12:W12,T12:T15,T12:Vx,T12:W12,T15:Vx,T15:W12,Vx:W12,Y
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,...,0.0,0.0,0.0,201.00,125.29,0.0,140.25,0.0,0.0,63.6
1,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,235.50,81.00,0.0,84.78,0.0,0.0,89.6
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,...,0.0,0.0,0.0,79.79,152.47,0.0,194.93,0.0,0.0,79.0


In [71]:
dfinter.to_csv(SORTIE + "dfinter.csv", index=False)
print("Sauvegarde OK :", SORTIE + "dfinter.csv")

Sauvegarde OK : ../Session3/dfinter.csv


## 6. `dffull.csv` — Variables + termes carrés et cubiques + interactions d'ordre 2

In [72]:
# On récupère uniquement les colonnes nouvelles de chaque bloc (sans Y ni doublons)
cols_base  = dfbase.columns.difference(["Y"])
cols_poly  = dfpoly.columns.difference(["Y"]).difference(cols_base)   # nouveautés poly
cols_inter = dfinter.columns.difference(["Y"]).difference(cols_base)  # nouveautés inter

dffull = pd.concat([
    dfbase[cols_base],    # effets principaux
    dfpoly[cols_poly],    # X² et X³ uniquement (pas de doublons)
    dfinter[cols_inter],  # interactions uniquement (pas de doublons)
    Y                     # variable cible
], axis=1)

print(f"dffull : {dffull.shape[0]} obs x {dffull.shape[1]-1} features + Y")
dffull.to_csv("dffull.csv", index=False)

dffull : 50 obs x 110 features + Y


## Récapitulatif

In [73]:
recap = pd.DataFrame({
    "Fichier"     : ["dfbase.csv", "dfpoly.csv", "dfinter.csv", "dffull.csv"],
    "Nb features" : [dfbase.shape[1]-1, dfpoly.shape[1]-1,
                     dfinter.shape[1]-1, dffull.shape[1]-1],
    "Contenu"     : [
        "Variables d'origine encodées",
        "dfbase + termes quadratiques et cubiques",
        "dfbase + interactions d'ordre 2",
        "dfbase + polynômes + interactions (complet)"
    ]
})
print(recap.to_string(index=False))

    Fichier  Nb features                                     Contenu
 dfbase.csv           13                Variables d'origine encodées
 dfpoly.csv           31    dfbase + termes quadratiques et cubiques
dfinter.csv           88             dfbase + interactions d'ordre 2
 dffull.csv          110 dfbase + polynômes + interactions (complet)
